# Colab Setup Test
Confirms: clone → deps → imports → EDA smoke test.
Run top to bottom on a fresh Colab session (Runtime → Disconnect and delete runtime first).


In [ ]:
import sys, os, subprocess, zipfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# ── 1. Repo ───────────────────────────────────────────────────────────────────
if IN_COLAB:
    if not os.path.exists("/content/DL_Proj"):
        !git clone https://github.com/fmssilva/DL_Proj.git /content/DL_Proj
    else:
        !git -C /content/DL_Proj pull --ff-only
    os.chdir("/content/DL_Proj/assignment_1")
    !pip install -r requirements.txt -q

# assignment_1/ must be on sys.path so 'import src.*' resolves directly from disk
ROOT = os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

# purge stale src.* cache — needed when re-running after a git pull
for k in [k for k in sys.modules if k == "src" or k.startswith("src.")]:
    del sys.modules[k]

print("cwd:", ROOT)

# ── 2. Data ───────────────────────────────────────────────────────────────────
# Locally: data/ already present. Colab: download from Google Drive.
_GDRIVE_FILE_ID = "1nVSQZxQubLEPXjSRqGn7rtPzkw-S0zIi"

if not Path("data/train_labels.csv").exists():
    if IN_COLAB:
        subprocess.run([sys.executable, "-m", "pip", "install", "gdown", "-q"], check=True)
        import gdown
        gdown.download(id=_GDRIVE_FILE_ID, output="data.zip", quiet=False)
        with zipfile.ZipFile("data.zip") as zf:
            zf.extractall("data")
        os.remove("data.zip")
        print("data/ ready")
    else:
        print("ERROR: data/ not found")
else:
    print("data/ already present")

# ── 3. Imports ────────────────────────────────────────────────────────────────
import torch
import pandas as pd
from src.config import SEED, CLASSES, DATA_DIR, create_output_dirs, set_seed
from src.datasets.dataset import get_train_val_loaders
from src.datasets.eda import class_distribution, check_data_integrity
from src.training.train import evaluate
from src.evaluation.metrics import compute_macro_f1
from src.models.mlp import MLP

set_seed(SEED)
create_output_dirs()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
df = pd.read_csv(DATA_DIR / "train_labels.csv")

print(f"Device: {device} | Rows: {len(df)} | Classes: {CLASSES}")
print("All imports OK.")

# ── 4. EDA smoke test ─────────────────────────────────────────────────────────
print("\n=== Class Distribution ===")
class_distribution(df)
print("\n=== Data Integrity ===")
valid, invalid = check_data_integrity(DATA_DIR / "Train", df)
print(f"{valid} valid, {invalid} invalid")
print("\nAll checks passed.")


cwd     : c:\Users\franc\OneDrive\Nossa_Pasta_2\5. Universidade\Cadeiras\DL\DL_Proj\assignment_1
sys.path: ['c:\\Users\\franc\\OneDrive\\Nossa_Pasta_2\\5. Universidade\\Cadeiras\\DL\\DL_Proj\\assignment_1', 'c:\\Users\\franc\\anaconda3\\envs\\cnn\\python310.zip', 'c:\\Users\\franc\\anaconda3\\envs\\cnn\\DLLs']
data/ already present — skipping download.
Device  : cpu
Classes : ['Bug', 'Fighting', 'Fire', 'Grass', 'Ground', 'Normal', 'Poison', 'Rock', 'Water']
Rows    : 3600
All good — src imports work.
--- sys.path (first 5) ---
  c:\Users\franc\OneDrive\Nossa_Pasta_2\5. Universidade\Cadeiras\DL\DL_Proj\assignment_1
  c:\Users\franc\anaconda3\envs\cnn\python310.zip
  c:\Users\franc\anaconda3\envs\cnn\DLLs
  c:\Users\franc\anaconda3\envs\cnn\lib
  c:\Users\franc\anaconda3\envs\cnn

--- src package location ---
  src: c:\Users\franc\OneDrive\Nossa_Pasta_2\5. Universidade\Cadeiras\DL\DL_Proj\assignment_1\src\__init__.py

--- sub-package probes ---
  OK   src.config                         